# Python API Example - Cargo Netbacks

This guide is designed to provide an example of how to call the Cargo Netbacks API endpoint, and store the data accordingly.

__N.B. This guide is just for Cargo Netbacks data. If you're looking for other API data products (such as Contracts or Freight routes), please refer to their according code example files.__ 

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/lng-cargo/netbacks.html

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html


In [ ]:
# Input the path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

## Reference Data endpoint

This query shows an overview on all available netbacks, showing all available ports and possible routes to/from these destinations (i.e. via Suez, Panama etc.).

An example of pulling all available Netbacks data for a specific route (e.g. Netbacks data for Sabine Pass via Suez) is shown later in this script. 

In [ ]:
# Define the function for listing all netback reference data
def list_netbacks(access_token):

    content = do_api_get_query(
        uri="/v1.0/netbacks/reference-data/", access_token=access_token
    )

    # retrieve release dates separately
    reldates = content["data"]["staticData"]["sparkReleases"]

    # return reference data
    refdata_dict = content["data"]

    return reldates, refdata_dict

# Fetch reference data:
reldates, refdata_dict = list_netbacks(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['staticData']['fobPorts'])
available_df.head(3)

## Defining data calling function

Details on the available parameters can be found on our API docs website:

https://www.sparkcommodities.com/api/lng-cargo/netbacks.html

__N.B.:__ This endpoint provides the option to return a JSON or CSV formatted response. Metadata is only available in the JSON format.

In [ ]:
from io import StringIO

def fetch_netback(access_token, fob_port_uuid, release=None, via_point=None, laden=None, ballast=None, start=None, end=None, percent_hire=None, format='json'):
    
    query_params = "?fob-port={}".format(fob_port_uuid)
    if release is not None:
        query_params += "&release-date={}".format(release)
    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    if via_point is not None:
        query_params += "&via-point={}".format(via_point)
    if laden is not None:
        query_params += "&laden-congestion-days={}".format(laden)
    if ballast is not None:
        query_params += "&ballast-congestion-days={}".format(ballast)
    if percent_hire is not None:
        query_params += "&percent-hire={}".format(percent_hire)
    
    
    content = do_api_get_query(
        uri="/v1.0/netbacks/{}".format(query_params),
        access_token=access_token, format=format,
    )
    
    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data

### Choosing FOB Port

In [ ]:
# Select Route
tsab = available_df[available_df["name"] == 'Sabine Pass']["uuid"].iloc[0]

### JSON Response

In [ ]:
## Calling data in JSON format

json_data = fetch_netback(access_token, fob_port_uuid=tsab, via_point='cogh', release='2026-07-06', format='json')
json_data

### CSV Response

Calling the data in CSV format & formatting as a Pandas DataFrame. Here, we also utilise the "start" and "end" parameters to call a range of historical data in one call.

In [ ]:
df_csv = fetch_netback(access_token, fob_port_uuid=tsab, via_point='cogh', start='2026-01-01', end='2026-06-30', format='csv')
df_csv

## N.B. Historical Data Limits

Currently, a maximum of 1 year's worth of historical data can be called at one time due to the size of the data file. 

If more data points are required, the below code can be used: the function calls the historical data 1 year at a time and combines the data into one Pandas DataFrame

In [ ]:

def loop_historical_data(access_token, fob_port_uuid=None, via_point=None, percent_hire=None, start=None, end=None):
    
    hist_diff = (datetime.datetime.strptime(end, '%Y-%m-%d') - datetime.datetime.strptime(start, '%Y-%m-%d')).days
    t = 0

    starts = []
    ends = []

    w = 365

    while t < hist_diff:
        # initialising dataframe
        if t == 0 and hist_diff>w:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
            hist_df = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=start, end=diff_end, 
                                    percent_hire=percent_hire, format='csv')

            starts.append(start)
            ends.append(diff_end)
        
        elif t==0 and hist_diff<=w:
            hist_df = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=start, end=end, 
                                    percent_hire=percent_hire, format='csv')

        # appending additional historical data
        else:
            if t < hist_diff-w:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
                historical_addition = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=diff_start, end=diff_end, 
                                    percent_hire=percent_hire, format='csv')
                
                try:
                    hist_df = pd.concat([hist_df,historical_addition])
                #exception if hist_df is empty
                except:
                    hist_df = historical_addition.copy()
                starts.append(start)
                ends.append(diff_end)
            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')

                historical_addition = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=diff_start, end=diff_end, 
                                    percent_hire=percent_hire, format='csv')

                hist_df = pd.concat([hist_df, historical_addition])
                starts.append(start)
                ends.append(diff_end)
        
        #looping by year
        t += w

    for c in list(hist_df.columns)[4:]:
        hist_df[c] = pd.to_numeric(hist_df[c])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])
    hist_df['LoadMonthDate'] = pd.to_datetime(hist_df['LoadMonth'])
    hist_df['CalMonth'] = hist_df['LoadMonthDate'].dt.strftime('%b%y')

    hist_df = hist_df.drop_duplicates()
    hist_df = hist_df.sort_values(['ReleaseDate', 'LoadDate'])

    return hist_df